# Phase 3 — Huấn luyện MobileNetV3-Small trên Colab

Chọn **Runtime → Change runtime type → T4 GPU**. Notebook chỉ chuẩn bị môi trường và gọi `src.train`; toàn bộ logic huấn luyện nằm trong mã nguồn dự án.

Trên Mac chạy `python -m src.package_colab`, rồi đưa `artifacts/phase3_colab_bundle.zip` vào `MyDrive/SeminarChuyenDe/`. Gói chỉ gồm mã nguồn, manifest và bộ ảnh TrashNet công khai. Checkpoint và history được lưu trong Drive để không mất khi Colab ngắt kết nối.

Không dùng tập test để điều chỉnh mô hình. Nếu Colab bị ngắt giữa chừng, dùng cell resume thay vì chạy mới vào cùng thư mục.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi
import torch, torchvision, sklearn
print('torch', torch.__version__, 'torchvision', torchvision.__version__, 'sklearn', sklearn.__version__)
assert torch.cuda.is_available(), 'Cần chọn runtime GPU trước khi huấn luyện'


In [ ]:
from pathlib import Path
import hashlib, os, zipfile
bundle = Path('/content/drive/MyDrive/SeminarChuyenDe/phase3_colab_bundle.zip')
expected_bundle_sha256 = '9b66c7c19b23a9ca12e67ceb67c5301fd05473d8d780f0269dc2a5d207d9ea3d'
assert bundle.is_file(), f'Không tìm thấy {bundle}'
assert hashlib.sha256(bundle.read_bytes()).hexdigest() == expected_bundle_sha256, 'Gói đã khác phiên bản; tạo lại notebook/checksum'
project = Path('/content/seminar')
project.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    z.extractall(project)
archive = project / 'data/raw/trashnet-dataset-resized.zip'
assert hashlib.sha256(archive.read_bytes()).hexdigest() == '0bf472790f8b20e5c950d5b5012a9d38af0d3392efd65f8ce171334fc16b07c2'
with zipfile.ZipFile(archive) as z:
    z.extractall(project / 'data/raw', members=[n for n in z.namelist() if n.startswith('dataset-resized/')])
manifest = project / 'data/processed/trashnet_manifest.csv'
assert hashlib.sha256(manifest.read_bytes()).hexdigest() == 'eea2338ba07941cd61abff4703734126b0275f4a51bad4d664d09ab4f63dd1de'
os.chdir(project)
print('Project/data ready:', project, 'images:', len(list((project/'data/raw/dataset-resized').rglob('*.jpg'))))


In [ ]:
!python -m src.check_data


## Chạy mới
Cell dưới tạo một run mới. Chỉ chạy khi thư mục `baseline_v1` chưa có checkpoint. Mặc định tối đa 8 epoch classifier + 25 epoch fine-tune, early stopping theo validation macro-F1.

In [ ]:
!python -m src.train --run-dir /content/drive/MyDrive/SeminarChuyenDe/runs/baseline_v1


## Chỉ dùng khi Colab ngắt giữa chừng
Cell này đọc `last.pt` và tiếp tục từ epoch kế tiếp; không chạy nếu lượt huấn luyện mới ở trên đã hoàn thành.

In [ ]:
# Bỏ dấu # ở dòng dưới nếu cần tiếp tục:
# !python -m src.train --run-dir /content/drive/MyDrive/SeminarChuyenDe/runs/baseline_v1 --resume


In [ ]:
from PIL import Image
from IPython.display import display
run = Path('/content/drive/MyDrive/SeminarChuyenDe/runs/baseline_v1')
print((run/'run_summary.json').read_text())
display(Image.open(run/'learning_curves.png'))
!python -m src.predict --checkpoint /content/drive/MyDrive/SeminarChuyenDe/runs/baseline_v1/best.pt --image data/raw/dataset-resized/glass/glass105.jpg
